# Parkinson's Disease Detection Using Biomedical Voice Features

**Team Members**
- Charmilkumar Vijaykumar Patel | SID: 20036845 | cpatel6@stevens.edu
- Yunyang Zhang | SID: 20043349 | yzhang102@stevens.edu

**Course:** AAI-551  
**Dataset:** [Parkinson's Data Set – UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/174/parkinsons)

---

## Overview

This notebook orchestrates the complete Parkinson's Disease Detection pipeline:

1. **Load & validate** the voice-measurement dataset (`VoiceDataset`)
2. **Summarise** the dataset and write a text report
3. **Pre-model visualisations** – class distribution, correlation heatmap, box plots
4. **Train & evaluate** a Random Forest classifier (`ParkinsonPredictor`)
5. **Post-model visualisations** – confusion matrix, feature importance, ROC curve
6. **Part-2 Python feature demonstrations**

All business logic lives in the `src/` modules; this notebook only calls those modules.

## 0 · Setup

Add the `src/` directory to `sys.path` so the relative imports inside the
source modules work correctly when the notebook is run from the project root.

In [ ]:
import sys
from pathlib import Path

# Make sure src/ modules are importable
project_root = Path().resolve()
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Project root : {project_root}")
print(f"src/ on path : {src_dir.exists()}")

## 1 · Import Modules

In [ ]:
# Standard library
import math
import time

# Third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

# Project modules
from config import (
    BOXPLOT_FILE,
    CONFUSION_MATRIX_FILE,
    CORRELATION_HEATMAP_FILE,
    DATA_FILE,
    FEATURE_IMPORTANCE_FILE,
    PLOT_FILE,
    ROC_CURVE_FILE,
    SUMMARY_FILE,
)
from dataset import VoiceDataset
from model import ParkinsonPredictor
from utils import (
    prediction_generator,
    save_dataset_summary,
    select_numeric_features,
    validate_file_path,
    validate_required_columns,
)
from visualization import (
    plot_confusion_matrix,
    plot_feature_boxplots,
    plot_feature_correlation_heatmap,
    plot_feature_importance,
    plot_roc_curve,
    plot_status_distribution,
)

print("All modules imported successfully.")

## 2 · Load Dataset (`VoiceDataset` class)

In [ ]:
# Instantiate and load
dataset = VoiceDataset(DATA_FILE)
df = dataset.load_data()

# Demonstrate __str__ and __len__ operator overloads
print("__str__:", dataset)
print("__len__:", len(dataset), "samples")

df.head()

## 3 · Dataset Summary (Data I/O)

In [ ]:
# save_dataset_summary writes a .txt file and returns a dict (uses time + math)
summary = save_dataset_summary(df, SUMMARY_FILE)
print("Summary written to:", SUMMARY_FILE)
print("Summary dict:", summary)

# Verify file contents
print("\n--- File contents ---")
print(SUMMARY_FILE.read_text())

## 4 · Visualizations

All plots are saved to `results/` and displayed inline for report use.

### 4.1 Class Distribution Bar Chart

Shows how many voice samples belong to the *Healthy* (0) and *Parkinson's* (1) classes.
The dataset is imbalanced — Parkinson's samples outnumber healthy ones — which is
important context for interpreting model metrics.

In [ ]:
plot_status_distribution(df, PLOT_FILE)
print("Saved:", PLOT_FILE)
display(Image(str(PLOT_FILE)))

### 4.2 Feature Correlation Heatmap

Pearson correlation matrix for all 22 voice features.  
Strongly correlated feature groups (deep red / deep blue) indicate redundancy
and are useful for understanding which measurements carry the same information.

In [ ]:
plot_feature_correlation_heatmap(df, CORRELATION_HEATMAP_FILE)
print("Saved:", CORRELATION_HEATMAP_FILE)
display(Image(str(CORRELATION_HEATMAP_FILE)))

### 4.3 Train the Model

We train the Random Forest here so that the box plots (§4.4) can be ordered by
actual feature importance instead of arbitrary column order.

In [ ]:
# Prepare scaled train/test splits
X_train, X_test, y_train, y_test = dataset.prepare_train_test_data()
print(f"Train samples: {len(X_train)}  |  Test samples: {len(X_test)}")

# Train
start = time.time()           # built-in time module
predictor = ParkinsonPredictor(n_estimators=100)
predictor.train(X_train, y_train)
elapsed = time.time() - start

print(predictor)              # __str__ overload
print(f"Training time: {elapsed:.4f} seconds")

### 4.4 Feature Box Plots

Box plots show the distribution of the top 10 most important features split by
class label.  Features with clearly separated boxes are the most discriminative
for distinguishing Parkinson's from healthy voice samples.

In [ ]:
# Sort feature columns by RF importance so the most informative features appear first
importances_dict = predictor.get_feature_importances(dataset.feature_columns)
sorted_features = sorted(importances_dict, key=importances_dict.get, reverse=True)

plot_feature_boxplots(df, sorted_features, BOXPLOT_FILE)
print("Saved:", BOXPLOT_FILE)
display(Image(str(BOXPLOT_FILE)))

## 5 · Model Evaluation

In [ ]:
# Evaluate
results = predictor.evaluate(X_test, y_test)

print(f"Accuracy  : {results['accuracy']:.4f}")
print("\nConfusion Matrix:")
print(results["confusion_matrix"])
print("\nClassification Report:")
print(results["classification_report"])

### 5.1 Confusion Matrix Heatmap

Visualises true positives, true negatives, false positives, and false negatives.
For a medical screening model, **false negatives** (missed Parkinson's cases) are
especially important to minimise.

In [ ]:
plot_confusion_matrix(results["confusion_matrix"], CONFUSION_MATRIX_FILE)
print("Saved:", CONFUSION_MATRIX_FILE)
display(Image(str(CONFUSION_MATRIX_FILE)))

### 5.2 Feature Importance Chart

Shows which voice measurements contributed most to the Random Forest model's
predictions (measured by mean decrease in Gini impurity).  High-importance features
are the most useful biomarkers for Parkinson's detection.

In [ ]:
feat_names = list(importances_dict.keys())
feat_vals  = np.array(list(importances_dict.values()))

plot_feature_importance(feat_names, feat_vals, FEATURE_IMPORTANCE_FILE)
print("Saved:", FEATURE_IMPORTANCE_FILE)
display(Image(str(FEATURE_IMPORTANCE_FILE)))

### 5.3 ROC Curve

The Receiver Operating Characteristic (ROC) curve plots the true positive rate
against the false positive rate at every classification threshold.  The **AUC**
(Area Under the Curve) summarises overall discrimination ability:
- AUC = 1.0 → perfect classifier
- AUC = 0.5 → random classifier (dashed baseline)

In [ ]:
y_proba = predictor.predict_proba(X_test)[:, 1]   # probability of Parkinson's class

plot_roc_curve(y_test, y_proba, ROC_CURVE_FILE)
print("Saved:", ROC_CURVE_FILE)
display(Image(str(ROC_CURVE_FILE)))

## 6 · Part 2 Feature Demonstrations

This section explicitly exercises each of the required Part-2 Python features.

### 6.1 `filter()` + `lambda` (special function)

In [ ]:
# select_numeric_features uses filter() + lambda internally.
# Here we show an explicit in-notebook usage as well.
all_cols = list(df.columns)
numeric_cols = list(filter(lambda c: pd.api.types.is_numeric_dtype(df[c]), all_cols))
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols[:5]} ...")

### 6.2 List Comprehension

In [ ]:
# Build feature list by excluding identifier and target columns.
excluded = {"name", "status"}
feature_cols = [c for c in numeric_cols if c not in excluded]
print(f"Feature columns ({len(feature_cols)}): {feature_cols[:5]} ...")

### 6.3 Built-in Modules: `time` and `math`

In [ ]:
# time – measure training duration (already shown above; repeated here for clarity)
timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
print("Current timestamp:", timestamp)

# math – compute minimum bits needed to index every sample
n_samples = len(df)
bits_needed = math.ceil(math.log2(n_samples))
print(f"Samples: {n_samples}  →  bits needed to index each sample: {bits_needed}")

### 6.4 Generator Function (`prediction_generator`)

In [ ]:
# Obtain raw predictions for the test set
test_preds = predictor.predict(X_test)

# prediction_generator is a generator function that lazily yields messages
print("First 5 sample predictions (via generator):")
for msg in prediction_generator(test_preds[:5]):
    print(" ", msg)

### 6.5 Set Operations

In [ ]:
# validate_required_columns uses set difference internally.
# Shown explicitly here too.
required = {"status", "MDVP:Fo(Hz)", "HNR"}
available = set(df.columns)
missing = required - available
extra   = available - required

print(f"Required  : {required}")
print(f"Missing   : {missing if missing else 'none'}")
print(f"Extra cols: {len(extra)} additional columns present")

### 6.6 `__name__` Guard

All `src/*.py` modules use `if __name__ == "__main__":` guards so they can be
imported without side-effects.  For example, `src/dataset.py` can be imported
by the notebook without any code running automatically.

## 7 · Operator Overload Demonstration

In [ ]:
# __add__: combine two predictors' estimator counts
predictor_b = ParkinsonPredictor(n_estimators=50)
combined_estimators = predictor + predictor_b
print(f"predictor ({predictor.n_estimators}) + predictor_b ({predictor_b.n_estimators})"
      f" = {combined_estimators} total estimators")

# __len__: number of rows in the dataset
print(f"Dataset row count via len(): {len(dataset)}")

## 8 · Exception Handling Demonstration

In [ ]:
# Scenario 1 – FileNotFoundError for a missing dataset
try:
    bad_dataset = VoiceDataset(Path("data/does_not_exist.csv"))
    bad_dataset.load_data()
except FileNotFoundError as exc:
    print("Caught FileNotFoundError:", exc)

# Scenario 2 – RuntimeError for predicting before training
try:
    untrained = ParkinsonPredictor()
    untrained.predict(X_test)
except RuntimeError as exc:
    print("Caught RuntimeError:", exc)

## 9 · Summary

| Requirement | Where implemented |
|---|---|
| Two meaningful classes + relationship | `VoiceDataset`, `ParkinsonPredictor` (composition) |
| Two meaningful functions | `validate_file_path`, `save_dataset_summary`, … |
| Advanced libraries | pandas, numpy, matplotlib, scikit-learn |
| Exception handling (≥ 2) | `FileNotFoundError`, `ValueError`, `RuntimeError` |
| Data I/O | Read CSV → write summary.txt + 6 plot files |
| Loops & if statements | Throughout `dataset.py`, `utils.py`, `model.py` |
| Mutable types | `list`, `dict`, `pd.DataFrame` |
| Immutable types | `str`, `tuple`, `int`, `float` |
| `__str__`, `__len__`, `__add__` | `VoiceDataset`, `ParkinsonPredictor` |
| `filter()` + `lambda` | `select_numeric_features()` in `utils.py` |
| List comprehension | `select_numeric_features()`, notebook cell 6.2 |
| Built-in modules (`time`, `math`) | `utils.py` |
| Generator function | `prediction_generator()` in `utils.py` |
| Set operations | `validate_required_columns()` in `utils.py` |
| `__name__ == "__main__"` | All `src/*.py` modules |
| Docstrings & comments | All classes and functions |
| Pytest tests | `tests/test_utils.py`, `test_dataset.py`, `test_model.py` |

### Plots generated

| File | Description |
|---|---|
| `results/status_distribution.png` | Class distribution bar chart |
| `results/feature_correlation_heatmap.png` | Pearson correlation heatmap of all features |
| `results/feature_boxplots.png` | Box plots of top-10 features by class |
| `results/confusion_matrix.png` | Confusion matrix heatmap |
| `results/feature_importance.png` | Random Forest feature importances |
| `results/roc_curve.png` | ROC curve with AUC score |